# معمل التجربة: قاعدة البيانات السحابية Pinecone Serverless

قاعدة بيانات Pinecone هي الأشهر في العالم لتطبيقات الذكاء الاصطناعي المؤسسية بدون الحاجة لإدارة سيرفرات أو بنية تحتية.

في المعمل ده:
1. هنفحص مفتاح Pinecone.
2. هنجرب كيفية تنظيم وفهرسة البيانات والبحث السريع فيها.

### 1. فحص الاتصال وتجهيز التضمينات
يلا نتأكد من التجهيزات:

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
pinecone_key = os.getenv("PINECONE_API_KEY")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"مفتاح Pinecone موجود؟ {bool(pinecone_key)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

مفتاح Pinecone موجود؟ False


### 2. تخزين الوثائق والبحث الدلالي
هنجرب البحث في Pinecone أو في المحاكي المحلي المتطابق معه برمجياً:

In [2]:
docs = [
    Document(page_content="تتميز Pinecone بتوفير فهارس Serverless تتوسع تلقائياً حسب حجم العمل.", metadata={"service": "pinecone"}),
    Document(page_content="تدعم Pinecone التصفية الفورية بالبيانات الوصفية بجانب البحث الدلالي.", metadata={"service": "pinecone"}),
]

if pinecone_key and len(pinecone_key) > 5:
    try:
        from langchain_pinecone import PineconeVectorStore
        store = PineconeVectorStore.from_documents(docs, embeddings, index_name="rag-index")
        print("متصل بقاعدة Pinecone السحابية الحية!")
    except Exception as e:
        print(f"ملاحظة اتصال ({e})، تشغيل المحاكي لضمان استمرار التجربة.")
        from langchain_core.vectorstores import InMemoryVectorStore
        store = InMemoryVectorStore.from_documents(docs, embeddings)
else:
    from langchain_core.vectorstores import InMemoryVectorStore
    store = InMemoryVectorStore.from_documents(docs, embeddings)
    print("تم تشغيل المحاكي المحلي لنظام Pinecone بنجاح.")

res = store.similarity_search("كيف تتوسع Pinecone تلقائياً؟", k=1)
print(f"النتيجة: {res[0].page_content}")

تم تشغيل المحاكي المحلي لنظام Pinecone بنجاح.
النتيجة: تتميز Pinecone بتوفير فهارس Serverless تتوسع تلقائياً حسب حجم العمل.


### 3. جرب بنفسك!
جرب تغير السؤال أو تضيف وثائق جديدة وتستكشف طريقة عمل البحث السحابي.